# Results

In [1]:
DATASET_PATH = "../dataset/dataset.json"
OUT_PATH      = "../results/bigcodebench_llm_clones.json" 
FILTERED_PATH = "../results/bigcodebench_llm_clones_filtered.json" 
FILTERED_CODEBLEU_PATH = "../results/bigcodebench_llm_clones_filtered_codebleu.json" 
FINAL_DATASET = "../results/bigcodebench_clone_dataset.json" 
codebleu_threshold = 0.4


## Cleaning code

In [2]:
import json
from src.utils import clean_code 

# Load the JSON file
with open(OUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Clean each code snippet
for entry in data:
    for clone in entry.get("clones", []):
        code = clone.get("code", "")
        if code:
            cleaned = clean_code(code)
            clone["code"] = cleaned  # save back to JSON

# Save back to the same JSON file (or a new file)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Cleaned code saved to {OUT_PATH}")


Cleaned code saved to ../results/bigcodebench_llm_clones.json


## Running tests

In [3]:
import json
from src.utils import install_package, extract_required_packages_clones

with open(OUT_PATH, "r", encoding="utf-8") as f:
    clone_data = json.load(f)  # dataset with clones

packages = extract_required_packages_clones(clone_data)
print(f"Required packages: {packages}")

for pkg in packages:
    try:
        __import__(pkg)
    except ImportError:
        try:
            print(f"Installing missing package: {pkg}")
            install_package(pkg)
        except Exception as e:
            # Catch all exceptions to prevent script from stopping
            print(f"⚠️ Could not install package {pkg}, skipping. Reason: {e}")
    except Exception as e:
        # Catch any unexpected import errors
        print(f"⚠️ Error importing package {pkg}, skipping. Reason: {e}")

print("✅ Finished checking/installing packages.")


Required packages: {'matplotlib', 'sklearn', 'scipy', 'pandas', 'numpy', 'hashlib'}
✅ Finished checking/installing packages.


In [4]:
import json 
import re
from src.utils import validate_with_unittest

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)  # original dataset with tests
with open(OUT_PATH, "r", encoding="utf-8") as f:
    clone_data = json.load(f)  # dataset with clones

data_by_id = {entry["id"]: entry for entry in data}

for i, clone_entry in enumerate(clone_data, 1):
    entry_id = clone_entry["id"]
    print(f"\nTesting Entry {i}/{len(clone_data)} | id={entry_id}")

    if entry_id not in data_by_id:
        print(f"  ⚠️ No matching entry found for {entry_id}, skipping.")
        continue

    entry = data_by_id[entry_id]
    tests_list = entry.get("test", [])

    for k, clone in enumerate(clone_entry.get("clones", []), 1):
        try:
            # Skip when it was already tested 
            if "test_results" in clone and clone["test_results"]:
                print(f"  Clone {k}: already has test results, skipping.")
                continue

            code = clone["code"]
            test_results = validate_with_unittest(code, tests_list)
            clone["test_results"] = test_results

            passed = sum(1 for v in test_results.values() if v == "PASS")
            failed = sum(1 for v in test_results.values() if v == "FAIL")
            error = sum(1 for v in test_results.values() if v.startswith("ERROR"))
            total = len(test_results)
            print(f"  Clone {k}: PASS={passed}, FAIL={failed}, ERROR={error}, Total={total}")

        except Exception as e:
            print(f"  ❌ Unexpected error testing clone {k}: {e}")
            error_results = {}
            for test_code in tests_list:
                match = re.search(r'def (\w+)\(', test_code)
                test_name = match.group(1) if match else f"unknown_test_{len(error_results)+1}"
                error_results[test_name] = "ERROR"
            clone["test_results"] = error_results
 
        with open(OUT_PATH, "w", encoding="utf-8") as f:
            json.dump(clone_data, f, indent=2)

print(f"\n✅ Done. Saved dataset with test results to {OUT_PATH}")



Testing Entry 1/12 | id=BigCodeBench/58
  Clone 1: PASS=5, FAIL=0, ERROR=0, Total=5

Testing Entry 2/12 | id=BigCodeBench/151
  Clone 1: PASS=4, FAIL=1, ERROR=0, Total=5

Testing Entry 3/12 | id=BigCodeBench/156
  Clone 1: PASS=0, FAIL=5, ERROR=0, Total=5

Testing Entry 4/12 | id=BigCodeBench/168
  Clone 1: PASS=5, FAIL=0, ERROR=0, Total=5

Testing Entry 5/12 | id=BigCodeBench/179
  Clone 1: PASS=3, FAIL=2, ERROR=0, Total=5

Testing Entry 6/12 | id=BigCodeBench/213
  Clone 1: PASS=4, FAIL=1, ERROR=0, Total=5

Testing Entry 7/12 | id=BigCodeBench/253
  Clone 1: PASS=0, FAIL=5, ERROR=0, Total=5

Testing Entry 8/12 | id=BigCodeBench/261
  Clone 1: PASS=5, FAIL=0, ERROR=0, Total=5

Testing Entry 9/12 | id=BigCodeBench/271
  Clone 1: PASS=3, FAIL=2, ERROR=0, Total=5

Testing Entry 10/12 | id=BigCodeBench/375
  Clone 1: PASS=5, FAIL=0, ERROR=0, Total=5

Testing Entry 11/12 | id=BigCodeBench/411
  Clone 1: PASS=5, FAIL=0, ERROR=0, Total=5

Testing Entry 12/12 | id=BigCodeBench/437
  Clone 1:

### Filtering dataset based on passed tests

In [5]:
import json

with open(OUT_PATH, "r", encoding="utf-8") as f:
    clone_data = json.load(f)

filtered_data = []

for entry in clone_data:
    # Keep only clones that passed all tests
    passing_clones = []
    for clone in entry.get("clones", []):
        if clone.get("test_results") and all(result == "PASS" for result in clone["test_results"].values()):
            clone_copy = clone.copy()
            clone_copy.pop("metrics", None)  # Remove metrics if present
            passing_clones.append(clone_copy)
    
    if passing_clones:  # Only keep entries with at least one passing clone
        filtered_entry = entry.copy()
        filtered_entry["clones"] = passing_clones
        filtered_data.append(filtered_entry)

# Save filtered dataset
with open(FILTERED_PATH, "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, indent=2)

print(f"✅ Filtered dataset saved to {FILTERED_PATH}, entries: {len(filtered_data)}")


✅ Filtered dataset saved to ../results/bigcodebench_llm_clones_filtered.json, entries: 6


## Calculating similarity scores

### Codebleu vs original code

In [6]:
import json
from codebleu import calc_codebleu 
from src.utils import remove_function_signature

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
with open(FILTERED_PATH, "r", encoding="utf-8") as f:
    clone_data = json.load(f)

data_by_id = {entry["id"]: entry for entry in data}

for i, clone_entry in enumerate(clone_data, 1):
    entry_id = clone_entry["id"]
    print(f"\nEvaluating Entry {i}/{len(clone_data)} | id={entry_id}")

    if entry_id not in data_by_id:
        print(f"  ⚠️ No matching entry found for {entry_id}, skipping.")
        continue

    entry = data_by_id[entry_id]
    ref_code = entry.get("original_code", None)
    if not ref_code:
        print("  ⚠️ No reference code found, skipping.")
        continue

    clones = clone_entry.get("clones", [])

    for idx, clone in enumerate(clones):
        clone.setdefault("metrics", {})
        clone["metrics"].setdefault("codebleu", {})

        try:
            # --- Ignore function signatures ---
            ref_body = remove_function_signature(ref_code)
            clone_body = remove_function_signature(clone["code"])

            score = calc_codebleu([ref_body], [clone_body], lang="python")
            clone["metrics"]["codebleu"]["originalcode"] = float(score["codebleu"])
            print(f"  Clone {idx + 1} vs original: CodeBLEU={score['codebleu']:.4f}")
        except Exception as e:
            print(f"  ❌ Error computing CodeBLEU for clone {idx + 1} vs original: {e}")
  
    with open(FILTERED_PATH, "w", encoding="utf-8") as f:
        json.dump(clone_data, f, indent=2) 

print(f"\n✅ Done. Final dataset with CodeBLEU scores (vs original, signature ignored) saved to {FILTERED_PATH}")



Evaluating Entry 1/6 | id=BigCodeBench/58
  Clone 1 vs original: CodeBLEU=0.5052

Evaluating Entry 2/6 | id=BigCodeBench/168
  Clone 1 vs original: CodeBLEU=0.3778

Evaluating Entry 3/6 | id=BigCodeBench/261
  Clone 1 vs original: CodeBLEU=0.7142

Evaluating Entry 4/6 | id=BigCodeBench/375
  Clone 1 vs original: CodeBLEU=0.3561

Evaluating Entry 5/6 | id=BigCodeBench/411
  Clone 1 vs original: CodeBLEU=0.4271

Evaluating Entry 6/6 | id=BigCodeBench/437
  Clone 1 vs original: CodeBLEU=0.2204

✅ Done. Final dataset with CodeBLEU scores (vs original, signature ignored) saved to ../results/bigcodebench_llm_clones_filtered.json


### Filter based on codebleu vs original

In [7]:
import json 

with open(FILTERED_PATH, "r", encoding="utf-8") as f:
    clone_data = json.load(f)

filtered_data = []

for entry in clone_data:
    clones = entry.get("clones", [])
    if not clones:
        continue

    final_clones = []
    for idx, clone in enumerate(clones):
        cb = clone.get("metrics", {}).get("codebleu", {})
        orig_score = float(cb.get("originalcode", 0.0))
        if orig_score <= codebleu_threshold:
            final_clones.append(clone)

    if final_clones:
        filtered_entry = {k: v for k, v in entry.items() if k != "clones"}
        filtered_entry["clones"] = final_clones
        filtered_data.append(filtered_entry)

# Save result
with open(FILTERED_CODEBLEU_PATH, "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, indent=2)

print(f"✅ Filtered dataset (by original) saved to {FILTERED_CODEBLEU_PATH}, entries: {len(filtered_data)}")


✅ Filtered dataset (by original) saved to ../results/bigcodebench_llm_clones_filtered_codebleu.json, entries: 3


### Between all clones 

In [8]:
import json
from codebleu import calc_codebleu
from itertools import combinations

with open(FILTERED_CODEBLEU_PATH, "r", encoding="utf-8") as f:
    clone_data = json.load(f)

for i, clone_entry in enumerate(clone_data, 1):
    entry_id = clone_entry["id"]
    print(f"\nEvaluating Entry {i}/{len(clone_data)} | id={entry_id}")

    clones = clone_entry.get("clones", [])
    n = len(clones)

    for idx1, idx2 in combinations(range(n), 2):
        clone1 = clones[idx1]
        clone2 = clones[idx2]

        clone1_id = clone1.get("clone_id", f"clone_{idx1 + 1}")
        clone2_id = clone2.get("clone_id", f"clone_{idx2 + 1}")

        clone1.setdefault("metrics", {}).setdefault("codebleu", {})
        clone2.setdefault("metrics", {}).setdefault("codebleu", {})

        # Skip if already computed in either direction
        if clone2_id in clone1["metrics"]["codebleu"] and clone1_id in clone2["metrics"]["codebleu"]:
            print(f"  Skipping Clone {idx1 + 1} vs Clone {idx2 + 1} (already computed)")
            continue

        try:
            score = calc_codebleu([clone1["code"]], [clone2["code"]], lang="python")
            val = float(score["codebleu"])
            clone1["metrics"]["codebleu"][clone2_id] = val
            clone2["metrics"]["codebleu"][clone1_id] = val

            print(f"  Clone {idx1 + 1} vs Clone {idx2 + 1}: CodeBLEU={val:.4f}")
        except Exception as e:
            print(f"  ❌ Error computing CodeBLEU for clones {idx1 + 1} vs {idx2 + 1}: {e}")

    # Save after finishing each entry
    with open(FILTERED_CODEBLEU_PATH, "w", encoding="utf-8") as f:
        json.dump(clone_data, f, indent=2)

print(f"\n✅ Done. Final dataset with CodeBLEU scores (clone vs clone) saved to {FILTERED_CODEBLEU_PATH}")



Evaluating Entry 1/3 | id=BigCodeBench/168

Evaluating Entry 2/3 | id=BigCodeBench/375

Evaluating Entry 3/3 | id=BigCodeBench/437

✅ Done. Final dataset with CodeBLEU scores (clone vs clone) saved to ../results/bigcodebench_llm_clones_filtered_codebleu.json
